# Lecture16: KuCoin ML/RL Pipeline (ARIMA reference)

??????? ???????? ?? ?????? `arima_sber_moex_pipeline`, ?? ??????????? ??? KuCoin (NEAR-USDT/NEARUSDTM) ? ???????????? ? RL-????????????.


## 1) ????????? ?????????

In [ ]:
!pip -q install pandas numpy requests statsmodels arch matplotlib

## 2) ??????? ? ?????????

ARIMA ? backtest ????????? ????????? ?????????: fit -> forecast -> ??????? -> ?????????.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from arch import arch_model
from statsmodels.tsa.arima.model import ARIMA

SPOT_SYMBOL = 'NEAR-USDT'
FUTURES_SYMBOL = 'NEARUSDTM'
CANDLE_TYPE = '1min'
ARIMA_ORDER = (1, 1, 2)
FORECAST_HORIZON = 5
HISTORY_LIMIT = 1200
BACKTEST_POINTS = 180


## 3) ???????? OHLCV ? KuCoin (REST)

In [ ]:
url = 'https://api.kucoin.com/api/v1/market/candles'
params = {'symbol': SPOT_SYMBOL, 'type': CANDLE_TYPE}
raw = requests.get(url, params=params, timeout=20).json()['data']

df = pd.DataFrame(raw, columns=['ts', 'open', 'close', 'high', 'low', 'volume', 'turnover'])
df = df.iloc[::-1].reset_index(drop=True)
for c in ['open', 'close', 'high', 'low', 'volume', 'turnover']:
    df[c] = df[c].astype(float)
df['ts'] = pd.to_datetime(df['ts'].astype(int), unit='s', utc=True)
df = df.tail(HISTORY_LIMIT).copy()

df[['ts', 'open', 'high', 'low', 'close']].tail(5)


## 4) ARIMA fit ? backtest overlay (??? ? ?????????)

?????????? `actual_price` ? `predicted_price`, ??????? MSE/MAE/MAPE.

In [ ]:
close = df['close'].copy()
fit = ARIMA(close, order=ARIMA_ORDER).fit()

start_idx = max(ARIMA_ORDER[1], len(close) - BACKTEST_POINTS)
pred_res = fit.get_prediction(start=start_idx, end=len(close)-1)
pred = pred_res.predicted_mean
conf = pred_res.conf_int(alpha=0.05)

forecast_df = pd.DataFrame({
    'actual_price': close.iloc[start_idx:].values,
    'predicted_price': pred.values,
    'lower_price': conf.iloc[:, 0].values,
    'upper_price': conf.iloc[:, 1].values,
}, index=df['ts'].iloc[start_idx:])

err = forecast_df['actual_price'] - forecast_df['predicted_price']
mse = float((err.pow(2)).mean())
mae = float(err.abs().mean())
mape = float((err.abs() / forecast_df['actual_price'].replace(0, np.nan)).dropna().mean())

print('ARIMA order:', ARIMA_ORDER)
print('Backtest points:', len(forecast_df))
print('MSE:', round(mse, 8))
print('MAE:', round(mae, 8))
print('MAPE:', round(mape, 8))
forecast_df.head()


## 5) ?????? actual vs predicted

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(forecast_df.index, forecast_df['actual_price'], label='Actual', color='black')
plt.plot(forecast_df.index, forecast_df['predicted_price'], label='Predicted', color='orange')
plt.fill_between(
    forecast_df.index,
    forecast_df['lower_price'],
    forecast_df['upper_price'],
    color='orange',
    alpha=0.15,
    label='95% CI'
)
plt.title('NEAR-USDT ARIMA Backtest Overlay')
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 6) Rolling ????????? ?? ????????? ???????? (??????????? ????)

?????? ??? ? ?????????: ???? `forecast_horizon` ???? ??????? ???? -> LONG, ????? SHORT.

In [ ]:
closes = close.reset_index(drop=True)
records = []

for t in range(200, len(closes) - FORECAST_HORIZON):
    hist = closes.iloc[: t + 1]
    f = ARIMA(hist, order=ARIMA_ORDER).fit()
    forecast_h = float(f.forecast(steps=FORECAST_HORIZON).iloc[-1])
    price_t = float(closes.iloc[t])
    price_next = float(closes.iloc[t + FORECAST_HORIZON])

    position = 1 if forecast_h > price_t else -1
    pnl = position * (price_next - price_t) / price_t

    records.append({
        't': t,
        'price_t': price_t,
        'price_next': price_next,
        'forecast_h': forecast_h,
        'position': position,
        'pnl': pnl,
    })

trades_df = pd.DataFrame(records)
trades_df['equity_curve'] = (1 + trades_df['pnl']).cumprod()
trades_df.tail(5)


## 7) ?????? ??? RL-????: ret_hat ? sigma_hat

`ret_hat` = ???-?????????? ?? ARIMA ????????,
`sigma_hat` = GARCH(1,1) ?????????????.

In [ ]:
last_fit = ARIMA(close, order=ARIMA_ORDER).fit()
forecast_price = float(last_fit.forecast(steps=FORECAST_HORIZON).iloc[-1])
spot_price = float(close.iloc[-1])

ret_hat = float(np.log(forecast_price / spot_price) / FORECAST_HORIZON)

ret_series = np.log(close).diff().dropna()
garch = arch_model(ret_series * 100, mean='Zero', vol='GARCH', p=1, q=1).fit(disp='off')
var_1 = float(garch.forecast(horizon=1).variance.iloc[-1, 0])
sigma_hat = float(max((var_1 ** 0.5) / 100, 1e-4))

z_score = float(ret_hat / max(sigma_hat, 1e-8))
direction = 'LONG' if forecast_price > spot_price else 'SHORT'

print('spot_price:', round(spot_price, 6))
print('forecast_price:', round(forecast_price, 6))
print('ret_hat:', round(ret_hat, 8))
print('sigma_hat:', round(sigma_hat, 8))
print('z_score:', round(z_score, 4))
print('direction:', direction)


## 8) ??????? ????????? ??? ????????????? RL-????

In [ ]:
state = {
    'timestamp': df['ts'].iloc[-1].isoformat(),
    'asset': 'NEAR',
    'spot_symbol': SPOT_SYMBOL,
    'futures_symbol': FUTURES_SYMBOL,
    'spot_price': spot_price,
    'futures_price': spot_price,
    'ret_hat': ret_hat,
    'sigma_hat': sigma_hat,
    'z_score': z_score,
    'signal_direction': direction,
    'signal_model': 'arima_garch_ref',
    'arima_order': list(ARIMA_ORDER),
    'forecast_horizon': FORECAST_HORIZON,
    'forecast_price': forecast_price,
    'backtest_mse': mse,
    'backtest_mae': mae,
    'backtest_mape': mape,
    'candle_type': CANDLE_TYPE,
}

out_dir = Path('reports/kucoin_rl')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'latest_forecast_signal_kucoin_rl.json'
out_path.write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding='utf-8')

print('JSON saved:', out_path)
state


## 9) ??????? ????????? ??? RL-???????????

In [ ]:
print('SHADOW:')
print('python run_trade_signal.py --mode shadow --config config/micro_near_v1_1m.json --state-json reports/kucoin_rl/latest_forecast_signal_kucoin_rl.json')
print()
print('LIVE:')
print('python run_trade_signal.py --run-real-order --config config/micro_near_v1_1m.json --state-json reports/kucoin_rl/latest_forecast_signal_kucoin_rl.json')
print()
print('FORCE BUY_BOTH:')
print('python run_trade_signal.py --run-real-order --config config/micro_near_v1_1m.json --state-json reports/kucoin_rl/latest_forecast_signal_kucoin_rl.json --force-action BUY_BOTH --spot-qty 0.1 --futures-contracts 1')
